# Regresión en TensorFlow: de entrenamiento manual a `tf.keras`



## Objetivos

1. Generar y preparar los datos con TensorFlow.
2. Construir una regresión lineal **sin usar `model.fit()`**.
3. Entender `tf.Variable`, `GradientTape`, función de pérdida y actualización de parámetros.
4. Reescribir exactamente el mismo problema con **`tf.keras`**.
5. Crear una **capa personalizada** mediante `tf.keras.layers.Layer`.
6. Crear un **modelo personalizado** mediante `tf.keras.Model`.
7. Comparar las distintas formas de construir el mismo modelo.

> La intención es separar claramente lo que TensorFlow hace "a mano" de lo que Keras automatiza.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)

tf.random.set_seed(42)
np.random.seed(42)

## 1. Generación de datos

Conservamos la idea del notebook original: generar puntos de una recta con ruido.

Los parámetros reales son:

- pendiente: `5.5`
- intercepto: `2.3`

El modelo tendrá que aprender estos valores únicamente observando los pares \((x,y)\).

In [ ]:
n = 300

x = tf.linspace(-10.0, 10.0, n)
w_true = tf.constant(5.5, dtype=tf.float32)
b_true = tf.constant(2.3, dtype=tf.float32)

noise = tf.random.normal((n,), mean=0.0, stddev=1.9)
y = w_true * x + b_true + noise

plt.figure(figsize=(7, 4))
plt.scatter(x.numpy(), y.numpy(), s=15, alpha=0.7, label="datos")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Datos sintéticos para regresión")
plt.legend()
plt.show()

## 2. División train / validation / test con `tf.data`

`tf.data.Dataset` permite construir pipelines de entrada. Aquí hacemos una división reproducible y después agrupamos los datos en mini-batches.

In [ ]:
indices = tf.random.shuffle(tf.range(n), seed=42)

n_train = int(0.70 * n)
n_val = int(0.15 * n)

idx_train = indices[:n_train]
idx_val = indices[n_train:n_train+n_val]
idx_test = indices[n_train+n_val:]

x_train, y_train = tf.gather(x, idx_train), tf.gather(y, idx_train)
x_val, y_val = tf.gather(x, idx_val), tf.gather(y, idx_val)
x_test, y_test = tf.gather(x, idx_test), tf.gather(y, idx_test)

BATCH_SIZE = 30

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=42)
    .batch(BATCH_SIZE)
)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE)

print("train:", len(x_train), "validation:", len(x_val), "test:", len(x_test))

# Parte A — TensorFlow "manual"

## 3. Parámetros entrenables

En lugar de crear una capa `Dense`, creamos directamente:

$w \quad \text{y} \quad b$

como objetos `tf.Variable`.

Una `tf.Variable` puede ser modificada durante el entrenamiento.

In [ ]:
w = tf.Variable(tf.random.normal(()), name="w")
b = tf.Variable(tf.random.normal(()), name="b")

print("w inicial:", w.numpy())
print("b inicial:", b.numpy())

## 4. Forward pass

La predicción del modelo es:
$\hat{y} = wx + b$

Esta función representa el **forward pass**.

In [ ]:
def linear_model(x):
    return w * x + b

## 5. Función de pérdida

Para regresión usamos **Mean Squared Error (MSE)**:
$MSE = \frac{1}{N}\sum_i(y_i-\hat{y}_i)^2$

In [ ]:
def mse_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

## 6. `GradientTape`: cálculo automático de gradientes

TensorFlow registra dentro de `tf.GradientTape()` las operaciones realizadas con variables entrenables.

Después podemos solicitar:

```python
dw, db = tape.gradient(loss, [w, b])
```

Estos gradientes indican cómo cambia la pérdida al modificar cada parámetro.

In [ ]:
learning_rate = 0.01

def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        y_pred = linear_model(x_batch)
        loss = mse_loss(y_batch, y_pred)

    dw, db = tape.gradient(loss, [w, b])

    # Gradient descent manual
    w.assign_sub(learning_rate * dw)
    b.assign_sub(learning_rate * db)

    return loss

## 7. Loop de entrenamiento manual

Aquí escribimos explícitamente los dos loops principales:

- loop sobre épocas;
- loop sobre mini-batches.

Esto es precisamente lo que después abstraerá `model.fit()`.

In [ ]:
EPOCHS = 80
train_history = []
val_history = []

for epoch in range(EPOCHS):
    batch_losses = []

    for xb, yb in train_ds:
        loss = train_step(xb, yb)
        batch_losses.append(float(loss))

    train_loss = np.mean(batch_losses)

    val_losses = []
    for xb, yb in val_ds:
        val_pred = linear_model(xb)
        val_losses.append(float(mse_loss(yb, val_pred)))

    val_loss = np.mean(val_losses)

    train_history.append(train_loss)
    val_history.append(val_loss)

    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(
            f"Epoch {epoch+1:3d} | "
            f"train={train_loss:7.3f} | val={val_loss:7.3f} | "
            f"w={w.numpy():6.3f} | b={b.numpy():6.3f}"
        )

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_history, label="train")
plt.plot(val_history, label="validation")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("TensorFlow manual: curva de aprendizaje")
plt.legend()
plt.show()

In [ ]:
y_pred_test_manual = linear_model(x_test)
test_mse_manual = mse_loss(y_test, y_pred_test_manual)

print("MSE test:", float(test_mse_manual))
print("w aprendido:", float(w))
print("b aprendido:", float(b))
print("w real:", float(w_true))
print("b real:", float(b_true))

In [ ]:
order = tf.argsort(x_test)
xs = tf.gather(x_test, order)
ys_pred = tf.gather(y_pred_test_manual, order)

plt.figure(figsize=(7, 4))
plt.scatter(x_test, y_test, label="test", alpha=0.7,c='orange')
plt.plot(xs, ys_pred, linewidth=2, label="predicción")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regresión aprendida: TensorFlow manual")
plt.legend()
plt.show()

# Parte B — El mismo modelo con `tf.keras`

La capa:

```python
Dense(1)
```

implementa esencialmente:

\[
y = xW+b
\]

Para una entrada de una sola característica y una salida, contiene exactamente dos parámetros entrenables:

- un peso;
- un bias.

In [ ]:
keras_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(1)
])

keras_model.summary()

## 8. `compile()`

Con Keras ya no tenemos que actualizar manualmente los parámetros.

Indicamos:

- `optimizer`: cómo actualizar los pesos;
- `loss`: qué función minimizar;
- `metrics`: qué métricas registrar.

In [ ]:
keras_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss="mse",
    metrics=[tf.keras.metrics.MeanSquaredError(name="mse")]
)

## 9. `fit()`

`model.fit()` reemplaza gran parte del loop que escribimos anteriormente.

In [ ]:
history = keras_model.fit(
    x_train[:, None],
    y_train,
    validation_data=(x_val[:, None], y_val),
    epochs=80,
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Entrenamiento finalizado.")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("tf.keras: curva de aprendizaje")
plt.legend()
plt.show()

In [ ]:
test_loss, test_mse = keras_model.evaluate(x_test[:, None], y_test, verbose=0)

kernel, bias = keras_model.layers[0].get_weights()

print("MSE test:", test_mse)
print("w aprendido:", kernel.item())
print("b aprendido:", bias.item())
print("w real:", float(w_true))
print("b real:", float(b_true))

# 10. Tres formas de definir modelos en Keras

Keras ofrece principalmente:

1. `Sequential`: arquitectura lineal.
2. Functional API: arquitecturas más generales.
3. Subclassing de `tf.keras.Model`: control completo mediante Python.

In [ ]:
# 1) Sequential
model_seq = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1)
])

# 2) Functional API
inputs = tf.keras.Input(shape=(1,), name="x")
h = tf.keras.layers.Dense(8, activation="relu", name="hidden")(inputs)
outputs = tf.keras.layers.Dense(1, name="prediction")(h)
model_func = tf.keras.Model(inputs=inputs, outputs=outputs)

model_func.summary()

# 3) Sequential API (subclassing)
model = Sequential()
model.add(tf.keras.Input((1,)))
model.add(tf.keras.layers.Dense(8, activation="relu"))
model.add(Dense(1))


# 11. Personalización de una capa

Una capa personalizada hereda de `tf.keras.layers.Layer`.

Los dos métodos más importantes son:

 __init__ , donde puede realizar todas las inicializaciones independientes de la entrada \
build , donde conoce las formas de los tensores de entrada y puede hacer el resto de la inicialización \
call , donde haces el cálculo de reenvío \
get_config() serializa la configuración de tu capa para que Keras pueda reconstruirla idénticamente cuando guardas y cargas un modelo \
A continuación construimos nuestra propia versión educativa de una capa `Dense`.

In [ ]:
class MyDense(tf.keras.layers.Layer):
    def __init__(self, units=1, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = int(input_shape[-1])

        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_dim, self.units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs, **kwargs):
        z = tf.matmul(inputs, self.kernel) + self.bias
        return self.activation(z) if self.activation is not None else z

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.units,
            "activation": tf.keras.activations.serialize(self.activation),
        })
        return config

In [ ]:
custom_layer = MyDense(3, activation="relu",name="custom_dense_layer")
example = tf.constant([[1.0, 2.0]])

print(custom_layer(example))
print("Pesos:")
for variable in custom_layer.trainable_variables:
    print(variable.name, variable.shape)

cfg = custom_layer.get_config()
print(cfg)

In [ ]:
class CustomDropout(tf.keras.layers.Layer):
    def __init__(self, rate=0.5, **kwargs):
        super().__init__(**kwargs)
        if not 0 <= rate < 1:
            raise ValueError(f"rate debe estar en [0, 1), recibido: {rate}")
        self.rate = rate

    def call(self, inputs, training=None):    
        if not training:
            return inputs
        
        # Máscara binaria con probabilidad (1 - rate) de mantener
        keep_prob = 1.0 - self.rate
        mask = tf.cast(
            tf.random.uniform(tf.shape(inputs)) < keep_prob,
            inputs.dtype
        )
        # Escalado invertido (inverted dropout)
        return inputs * mask / keep_prob

    def get_config(self):
        config = super().get_config()
        config.update({"rate": self.rate})
        return config

In [ ]:
layer_dropout = CustomDropout(rate=0.3)
cfg = layer_dropout.get_config()
print(cfg)

In [ ]:

example = tf.constant([[[1.0, 2.0]]])

inputs = tf.keras.Input(shape=(1,2), name="x")
custom_layer1 = MyDense(3, activation="relu",name="custom_dense_layer1")(inputs)
layer_dropout = CustomDropout(rate=0.3)(custom_layer1, training=False)
custom_layer2 = MyDense(3, activation="relu",name="custom_dense_layer2")(layer_dropout)
model_func = tf.keras.Model(inputs=inputs, outputs=custom_layer2)

In [ ]:
model_func(example)

## ¿Por qué crear una capa personalizada?

Es útil cuando la operación deseada no corresponde directamente a una capa estándar. Por ejemplo:

- transformaciones físicas;
- operaciones geométricas;
- mecanismos de atención especiales;
- restricciones sobre parámetros;
- capas que combinan varias operaciones.

# 12. Personalización de un modelo

Un modelo personalizado hereda de `tf.keras.Model`.

En `__init__` definimos las capas y en `call()` especificamos cómo se conectan.

In [ ]:
class RegressionMLP(tf.keras.Model):
    def __init__(self, hidden_units=16):
        super().__init__()
        self.hidden = MyDense(hidden_units)
        self.output_layer = MyDense(1)

    def call(self, inputs, training=False):
        x = self.hidden(inputs)
        return self.output_layer(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.hidden,
        })
        return config

In [ ]:
custom_model = RegressionMLP(hidden_units=16)

# Construimos el modelo haciendo una primera llamada.
_ = custom_model(tf.zeros((1, 1)))

custom_model.summary()

custom_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="mse"
)

custom_history = custom_model.fit(
    x_train[:, None],
    y_train,
    validation_data=(x_val[:, None], y_val),
    epochs=40,
    batch_size=BATCH_SIZE,
    verbose=0
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(custom_history.history["loss"], label="train")
plt.plot(custom_history.history["val_loss"], label="validation")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Modelo personalizado")
plt.legend()
plt.show()

# 13. Relación entre implementación manual y Keras

| Concepto | TensorFlow manual | `tf.keras` |
|---|---|---|
| Parámetros | `tf.Variable` | pesos internos de las capas |
| Forward | función Python | `Layer.call()` / `Model.call()` |
| Derivadas | `GradientTape` | manejadas durante `fit()` |
| Actualización | `assign_sub()` | `optimizer` |
| Pérdida | función propia | `loss=` |
| Épocas/batches | loops explícitos | `model.fit()` |
| Inferencia | llamar función | `model.predict()` o `model(x)` |

La API de alto nivel **no cambia los fundamentos**. Automatiza el entrenamiento y organiza mejor el código.

# 14. Funciones de activación en TensorFlow/Keras

Las funciones de activación introducen transformaciones no lineales. Sin ellas, una composición de capas densas seguiría representando una transformación lineal.

Algunas de las más utilizadas son:

| Activación | Rango aproximado | Uso habitual |
|---|---:|---|
| `linear` | \((-\infty,\infty)\) | salida de regresión |
| `relu` | \([0,\infty)\) | capas ocultas |
| `leaky_relu` | \((-\infty,\infty)\) | alternativa a ReLU |
| `elu` | \((-\infty,\infty)\) | capas ocultas |
| `selu` | \((-\infty,\infty)\) | redes auto-normalizadas bajo condiciones específicas |
| `tanh` | \([-1,1]\) | activaciones centradas en cero |
| `sigmoid` | \([0,1]\) | probabilidad binaria |
| `softmax` | probabilidades que suman 1 | clasificación multiclase |

Para **regresión lineal**, la salida debe permanecer lineal.

In [ ]:
inputs = tf.linspace(-6.0, 6.0, 300)

activations = {
    "ReLU": tf.keras.activations.relu(inputs),
    "ELU": tf.keras.activations.elu(inputs),
    "Tanh": tf.keras.activations.tanh(inputs),
    "Sigmoid": tf.keras.activations.sigmoid(inputs),
    "Linear": tf.keras.activations.linear(inputs),
    "Swish": tf.keras.activations.swish(inputs),
    "LeakyReLU": tf.keras.activations.leaky_relu(inputs)
}

plt.figure(figsize=(9, 5))
for name, values in activations.items():
    plt.plot(inputs.numpy(), values.numpy(), label=name)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("x")
plt.ylabel("activation(x)")
plt.title("Funciones de activación")
plt.ylim(-2, 2)
plt.ylim(-2, 2)
plt.legend()
plt.show()

In [ ]:
z = tf.constant([[1.0, 1.2, 1.8, 2.0],
                 [3.0, 4.2, 5.8, 9.0]])

softmax_output = tf.keras.activations.softmax(z, axis=1)

print("Softmax:")
print(softmax_output.numpy())
print("\nSuma por fila:")
print(tf.reduce_sum(softmax_output, axis=1).numpy())

# 16. `compile()` en Keras, con más detalle

`compile()` configura **cómo se entrenará** el modelo. No ejecuta todavía ningún entrenamiento.

Los tres elementos principales son:

```python
model.compile(
    optimizerString (name of optimizer) keras.optimizers.
    loss=May be a string (name of loss function), or a keras.losses.Loss instance,
    loss_weights= list or dictionary specifying scalar coefficients (Python floats) to weight the loss contributions of different model outputs
    metrics=List of metrics to be evaluated by the model during training and testing
    weighted_metrics=List of metrics to be evaluated and weighted by sample_weight or class_weight during training and testing
)
```

### `optimizer`

Define cómo se actualizan los parámetros a partir de los gradientes.

Ejemplos:

```python
tf.keras.optimizers.SGD(...)
tf.keras.optimizers.Adam(...)
tf.keras.optimizers.RMSprop(...)
```

### `loss`

Es la función objetivo que Keras minimiza durante el entrenamiento.

Para regresión:

```python
loss="mse"
loss=tf.keras.losses.MeanSquaredError()
loss=tf.keras.losses.MeanAbsoluteError()
```

### `metrics`

Son medidas adicionales que queremos observar. No necesariamente se usan para actualizar los parámetros.

Ejemplo:

```python
metrics=[
    tf.keras.metrics.MeanAbsoluteError(name="mae"),
    tf.keras.metrics.RootMeanSquaredError(name="rmse")
]
```

In [ ]:
compiled_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

compiled_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse")
    ],
)

compiled_model.summary()

## Parámetros importantes de `compile()`

- `optimizer`: algoritmo de optimización.
- `loss`: función a minimizar.
- `metrics`: métricas mostradas durante entrenamiento/evaluación.
- `loss_weights`: ponderación de pérdidas en modelos con múltiples salidas.
- `run_eagerly`: útil para depuración; normalmente permanece en `False`.
- `jit_compile`: en versiones compatibles puede activar compilación acelerada.

# 17. `fit()` en Keras, con más detalle

`fit()` ejecuta el ciclo de entrenamiento.

```python
history = model.fit(
    x,
    y,
    batch_size=...,
    epochs=...,
    validation_data=...,
    callbacks=...,
    shuffle=True,
    verbose=1
)
```

### Argumentos especialmente importantes

- `x`, `y`: datos de entrada y objetivos.
- `batch_size`: número de muestras por actualización.
- `epochs`: número de recorridos completos sobre los datos.
- `validation_data`: conjunto usado para calcular métricas de validación.
- `validation_split`: separa una fracción de los arrays entregados a `fit()`.
- `shuffle`: reorganiza las muestras antes de cada época.
- `callbacks`: objetos que ejecutan acciones durante el entrenamiento.
- `verbose`: `0`, `1` o `2`.
- `initial_epoch`: permite continuar un entrenamiento desde una época concreta.
- `steps_per_epoch`: especialmente útil con datasets generadores o pipelines.

In [ ]:
demo_history = compiled_model.fit(
    x_train[:, None],
    y_train,
    batch_size=32,
    epochs=2,
    validation_data=(x_val[:, None], y_val),
    shuffle=True,
    verbose=1
)

# 18. El objeto `History`

`fit()` devuelve un objeto `History`.

Su atributo principal:

```python
history.history
```

es un diccionario cuyas claves corresponden a la pérdida y a las métricas registradas.

In [ ]:
print(demo_history.history.keys())

for key, values in demo_history.history.items():
    print(f"{key:10s} -> {len(values)} valores")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(demo_history.history["loss"], label="loss")
plt.plot(demo_history.history["val_loss"], label="val_loss")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Información almacenada por History")
plt.legend()
plt.show()

# 19. Callbacks en Keras

Un callback ejecuta acciones en momentos concretos del entrenamiento:

- inicio/final de entrenamiento;
- inicio/final de época;
- inicio/final de batch.

Aquí utilizaremos callbacks que permiten:

- guardar checkpoints;
- modificar el learning rate;
- escribir un archivo CSV;
- registrar información para TensorBoard;
- detener el proceso si aparecen valores numéricos inválidos.

In [ ]:
from pathlib import Path

callback_dir = Path("resultados_regresion")
callback_dir.mkdir(exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(callback_dir / "mejor_modelo.keras"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.LambdaCallback(
        on_train_batch_begin=lambda batch,logs: print(f'holaa {batch}'),
        on_epoch_end=lambda epoch, logs: print(f"Epoch {epoch} completed...")
    ),

    tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=0,
    verbose=0,
    mode='auto',
    restore_best_weights=False,
    start_from_epoch=0
),

    tf.keras.callbacks.TensorBoard(
        log_dir=str(callback_dir / "logs")
    ),

    tf.keras.callbacks.TerminateOnNaN()
]



## Crear un callback personalizado
##Podemos heredar de `tf.keras.callbacks.Callback`.

class PrintLearningRate(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        loss = None if logs is None else logs.get("loss")
        print(f"Epoch={epoch+1:02d} | lr={lr:.6f} | loss={loss:.4f}")

custom_callback = PrintLearningRate()

callbacks.append(custom_callback)

callback_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

callback_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

history_callbacks = callback_model.fit(
    x_train[:, None],
    y_train,
    validation_data=(x_val[:, None], y_val),
    epochs=5,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# 20. `evaluate()` y `predict()`

## `evaluate()`

Calcula la pérdida y las métricas configuradas en `compile()` sobre un conjunto de datos.

## `predict()`

Ejecuta inferencia y devuelve las salidas del modelo.

In [ ]:
results = callback_model.evaluate(
    x_test[:, None],
    y_test,
    batch_size=32,
    verbose=0,
    return_dict=True
)

print("evaluate():")
print(results)

In [ ]:
predictions = callback_model.predict(
    x_test[:, None],
    batch_size=32,
    verbose=0
)

print("Shape:", predictions.shape)
print("Primeras 5 predicciones:")
print(predictions[:5].ravel())

También puede llamarse directamente al modelo:

```python
y_pred = model(x, training=False)
```

Esta forma es especialmente cómoda dentro de código TensorFlow.

In [ ]:
direct_predictions = callback_model(x_test[:5, None], training=False)
print(direct_predictions.numpy().ravel())

# 21. Guardar y cargar modelos

El formato moderno recomendado para un modelo Keras completo es `.keras`.

Guarda:

- arquitectura;
- parámetros;
- configuración de compilación;
- estado del optimizador, cuando corresponde.

In [ ]:
model_path = "regression_model.keras"

callback_model.save(model_path)

loaded_model = tf.keras.models.load_model(model_path)

loaded_results = loaded_model.evaluate(
    x_test[:, None],
    y_test,
    verbose=0,
    return_dict=True
)

print("Modelo cargado:", loaded_results)

## Guardar únicamente los pesos

Esto es útil cuando la arquitectura ya está definida en código.

In [ ]:
weights_path = "regression_model.weights.h5"

callback_model.save_weights(weights_path)

new_model_same_architecture = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

new_model_same_architecture.load_weights(weights_path)

print(
    new_model_same_architecture(x_test[:3, None], training=False).numpy().ravel()
)
new_model_same_architecture.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)
new_model_same_architecture.evaluate(
    x_test[:, None],
    y_test,
    verbose=0,
    return_dict=True
)

## - L2 Regularizacion

Permite evitar overfitting. Se realiza a traves de la modificacion de la funcion de costo:
- **clasificacion**
$$J = -\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{-\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} }_\text{cross-entropy} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularizacion costo} \tag{2}$$

- **regresion**
$$J = \frac{1}{m} \sum\limits_{i = 1}^{m}  \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 }_\text{mse} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularizacion costo} \tag{2}$$

## - L1 Regularizacion

Permite evitar overfitting. Se realiza a traves de la modificacion de la funcion de costo:
- **clasificacion**
$$J = -\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{-\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} }_\text{cross-entropy} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]} }_\text{L2 regularizacion costo} \tag{2}$$

- **regresion**
$$J = \frac{1}{m} \sum\limits_{i = 1}^{m}  \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 }_\text{mse} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]} }_\text{L2 regularizacion costo} \tag{2}$$

## - L1-L2 Regularizacion

Permite evitar overfitting. Se realiza a traves de la modificacion de la funcion de costo:
- **clasificacion**
$$J = -\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{-\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} }_\text{cross-entropy} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]}+\frac{1}{m} \frac{\lambda_1}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularizacion costo} \tag{2}$$

- **regresion**
$$J = \frac{1}{m} \sum\limits_{i = 1}^{m}  \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 \tag{1}$$
modificada a:
$$J_{regul} = \small \underbrace{\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)} -a^{[L](i)}\large{)}^2 }_\text{mse} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]} +\frac{1}{m} \frac{\lambda_1}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2}}_\text{L2 regularizacion costo} \tag{2}$$

In [ ]:
from tensorflow.keras import regularizers

regularized_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),

    tf.keras.layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizers.l1_l2(
            l1=1e-5,
            l2=1e-4
        )
    ),

    tf.keras.layers.Dense(1)
])

regularized_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse"
)

regularized_model.summary()

In [ ]:
# Las pérdidas de regularización aparecen en model.losses
_ = regularized_model(tf.zeros((1, 1)))

print("Términos de regularización:")
for regularization_loss in regularized_model.losses:
    print(regularization_loss)

# 23. Dropout

`Dropout(rate)` anula aleatoriamente una fracción de las activaciones durante el modo de entrenamiento.

```python
tf.keras.layers.Dropout(0.2)
```

significa que aproximadamente el 20% de las activaciones se ponen a cero en cada llamada de entrenamiento.

Durante inferencia (`training=False`) Dropout no elimina unidades.

In [ ]:
dropout = tf.keras.layers.Dropout(0.5)

sample = tf.ones((2, 8))

print("training=True")
print(dropout(sample, training=True).numpy())

print("\ntraining=False")
print(dropout(sample, training=False).numpy())

# 24. Batch Normalization

`BatchNormalization` normaliza usando estadísticas calculadas a partir del batch durante entrenamiento.

Mantiene además estadísticas móviles que se utilizan durante inferencia.

Es frecuente encontrar arquitecturas del tipo:

```python
Dense -> BatchNormalization -> Activation
```

In [ ]:
bn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(32, use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(1)
])

bn_model.summary()

# 25. Layer Normalization

`LayerNormalization` normaliza cada observación a través de sus características.

A diferencia de Batch Normalization, no depende de las estadísticas del resto de ejemplos del batch.

Es muy utilizada en arquitecturas como Transformers.

In [ ]:
layer_norm = tf.keras.layers.LayerNormalization(axis=-1)

example = tf.constant([
    [1.0, 2.0, 8.0, 4.0],
    [10.0, 11.0, 13.0, 20.0]
])

print(layer_norm(example).numpy())

# 26. Group Normalization

`GroupNormalization` divide los canales/características en grupos y normaliza cada grupo independientemente.

Es especialmente útil cuando se trabaja con representaciones con múltiples canales y batches pequeños.

En versiones recientes de Keras:

In [ ]:
# Ejemplo con 8 características divididas en 4 grupos.
group_norm = tf.keras.layers.GroupNormalization(
    groups=4,
    axis=-1
)

example = tf.random.normal((3, 8))
normalized = group_norm(example)

print("Entrada:", example.shape)
print("Salida:", normalized.shape)

# 27. Comparación conceptual de normalizaciones

| Capa | Estadísticas utilizadas | Dependencia del batch |
|---|---|---|
| `BatchNormalization` | por característica/canal sobre el batch | Sí durante entrenamiento |
| `LayerNormalization` | sobre características de cada muestra | No |
| `GroupNormalization` | sobre grupos de características/canales | No |

La elección depende de la arquitectura, las dimensiones de los tensores y el tamaño de batch.

# 28. Modelo que combina varios elementos

El siguiente ejemplo muestra cómo combinar en una misma arquitectura:

- activaciones;
- L1/L2;
- Batch Normalization;
- Dropout.

No significa que todas las arquitecturas deban usar todos estos componentes simultáneamente.

In [ ]:
complete_model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),

    tf.keras.layers.Dense(
        64,
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l1_l2(
            l1=1e-6,
            l2=1e-4
        )
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.LayerNormalization(),

    tf.keras.layers.Dense(1)
])

complete_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

complete_model.summary()